In [ ]:
from collections import deque
import cv2
import torch
from torchvision.models.detection import ssd300_vgg16

# ── Load SSD model ────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3
model.load_state_dict(torch.load("./", map_location=device))
model.to(device)
model.eval()

# ── SSD Inference ─────────────────────────────────────────────────────────────

def ssd_predict(frame, conf_threshold=0.3):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    img_tensor = torch.tensor(img_rgb / 255.0, dtype=torch.float32)\
        .permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)[0]

    boxes = outputs["boxes"].cpu()
    labels = outputs["labels"].cpu()
    scores = outputs["scores"].cpu()

    people = []
    bags = []

    for box, label, score in zip(boxes, labels, scores):
        if score < conf_threshold:
            continue

        det = {
            "box": box.tolist(),
            "conf": float(score)
        }

        if int(label) == 1:
            people.append(det)
        elif int(label) == 2:
            bags.append(det)

    return people, bags

# ── Drawing ───────────────────────────────────────────────────────────────────

def draw_detections(frame, people, bags, stable_bags):
    # Draw people
    for p in people:
        x1, y1, x2, y2 = map(int, p["box"])
        conf = p["conf"]

        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 0), 2)
        cv2.putText(frame, f"person {conf:.2f}",
                    (x1, max(y1-8, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

    # Draw bags
    for b in bags:
        x1, y1, x2, y2 = map(int, b["box"])
        conf = b["conf"]

        color = (0, 0, 255) if stable_bags else (0, 165, 255)

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"bag {conf:.2f}",
                    (x1, max(y1-8, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Stats
    cv2.putText(frame,
                f"People: {len(people)}  Bags: {len(bags)}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Alert
    if stable_bags:
        cv2.putText(frame, "!! BAG DETECTED !!",
                    (10, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)

    return frame

# ── Webcam loop ───────────────────────────────────────────────────────────────

cap = cv2.VideoCapture("../../IMG_2162.mp4")

if not cap.isOpened():
    print("Error: Could not open webcam")
    exit()

# Temporal smoothing (reduces flickering)
bag_history = deque(maxlen=5)

print("Running SSD only... press 'q' to quit")
frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # frame = cv2.resize(frame, (300, 300)) # Resizes window to smaller size for faster processing (optional)
    people, bags = ssd_predict(frame, conf_threshold=0.4)
    
    # Speeds up video
    # % 2 → 2x faster
    # % 3 → 3x faster
    # % 5 → 5x faster
    # frame_count += 1
    # if frame_count % 2 != 0:
    #     continue
        
        

    # Temporal filtering
    bag_history.append(len(bags) > 0)
    stable_bags = sum(bag_history) >= 3

    annotated = draw_detections(frame.copy(), people, bags, stable_bags)

    cv2.imshow("SSD Detection", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()